# MindSync Stage 2: Audio (wav2vec2-Large) on RAVDESS

**Goal:** Train MindSyncAudioModel (wav2vec2-large-960h + classifier head) on RAVDESS speech audio, mapped to 4 emotion clusters.

**Runtime:** Free Colab T4 (use a Google account with fresh quota).

**Expected:** ~2-3 hours, ~70-80% test accuracy (RAVDESS is cleaner than GoEmotions text).

**Steps before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. 🔑 Secrets (key icon left sidebar) → Add **HF_TOKEN** = your Hugging Face token (write access)
3. Runtime → Run all

## 1 · Mount Drive & install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/mindsync')
ROOT.mkdir(exist_ok=True)
CKPT_DIR = ROOT / 'audio_ckpt'
CKPT_DIR.mkdir(exist_ok=True)
print('Checkpoints will save to:', CKPT_DIR)

In [ ]:
!pip install -q transformers==4.45.2 datasets==2.21.0 librosa==0.10.2 soundfile scikit-learn huggingface_hub==0.25.2
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
from google.colab import userdata
import os
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN secret missing — add it in 🔑 Secrets sidebar'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
print('HF token loaded ✓')

## 2 · Download RAVDESS (~200 MB from Zenodo)

In [ ]:
import pathlib, urllib.request, zipfile, os

RAVDESS_URL = 'https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip'
DATA_DIR = ROOT / 'ravdess'
DATA_DIR.mkdir(exist_ok=True)
ZIP = DATA_DIR / 'ravdess.zip'

if not ZIP.exists() or ZIP.stat().st_size < 100_000_000:
    print('Downloading RAVDESS (~200 MB)...')
    urllib.request.urlretrieve(RAVDESS_URL, ZIP)
    print(f'Saved: {ZIP} ({ZIP.stat().st_size / 1024**2:.1f} MB)')
else:
    print(f'Already downloaded: {ZIP} ({ZIP.stat().st_size / 1024**2:.1f} MB)')

AUDIO_ROOT = DATA_DIR / 'extracted'
if not AUDIO_ROOT.exists() or len(list(AUDIO_ROOT.rglob('*.wav'))) < 1000:
    print('Extracting...')
    AUDIO_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(AUDIO_ROOT)
wavs = sorted(AUDIO_ROOT.rglob('*.wav'))
print(f'Total wav files: {len(wavs)}')
print('Example:', wavs[0].name if wavs else 'NONE')

## 3 · Parse filenames → 4-cluster labels

RAVDESS filename format: `MM-VV-EE-II-SS-RR-AA.wav` where EE is emotion code:
- 01 neutral → Ambiguity
- 02 calm → Resilience
- 03 happy → Resilience
- 04 sad → Distress
- 05 angry → Aggression
- 06 fearful → Distress
- 07 disgust → Aggression
- 08 surprised → Ambiguity

In [ ]:
CLUSTER_NAMES = ['Distress', 'Resilience', 'Aggression', 'Ambiguity']
EMOTION_TO_CLUSTER = {
    1: 3,  # neutral → Ambiguity
    2: 1,  # calm → Resilience
    3: 1,  # happy → Resilience
    4: 0,  # sad → Distress
    5: 2,  # angry → Aggression
    6: 0,  # fearful → Distress
    7: 2,  # disgust → Aggression
    8: 3,  # surprised → Ambiguity
}

samples = []  # (path, cluster, actor_id)
for p in wavs:
    parts = p.stem.split('-')
    if len(parts) != 7:
        continue
    emotion = int(parts[2])
    actor   = int(parts[6])
    cluster = EMOTION_TO_CLUSTER[emotion]
    samples.append((str(p), cluster, actor))

from collections import Counter
print(f'Parsed: {len(samples)} samples')
print('Per-cluster counts:')
for c, n in sorted(Counter(s[1] for s in samples).items()):
    print(f'  {CLUSTER_NAMES[c]}: {n}')
print(f'Actors found: {sorted(set(s[2] for s in samples))}')

In [ ]:
# Speaker-independent split: actors 1-18 train, 19-21 val, 22-24 test
# Prevents speaker leakage (model memorising voices instead of emotions)
import random
random.seed(42)

train = [s for s in samples if s[2] <= 18]
val   = [s for s in samples if 19 <= s[2] <= 21]
test  = [s for s in samples if s[2] >= 22]

random.shuffle(train)
print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')
print('Train cluster dist:', dict(sorted(Counter(s[1] for s in train).items())))
print('Val   cluster dist:', dict(sorted(Counter(s[1] for s in val).items())))
print('Test  cluster dist:', dict(sorted(Counter(s[1] for s in test).items())))

## 4 · Audio dataset + DataLoader

In [ ]:
import librosa, numpy as np, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

TARGET_SR   = 16_000   # wav2vec2 expects 16 kHz
MAX_SAMPLES = 80_000   # 5 seconds — covers most RAVDESS utterances

class RavdessDataset(Dataset):
    def __init__(self, items, max_samples=MAX_SAMPLES):
        self.items = items
        self.max_samples = max_samples

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, cluster, _ = self.items[idx]
        wav, _ = librosa.load(path, sr=TARGET_SR, mono=True)
        # Trim silence + pad/truncate to fixed length
        wav, _ = librosa.effects.trim(wav, top_db=30)
        if len(wav) > self.max_samples:
            start = (len(wav) - self.max_samples) // 2
            wav = wav[start:start + self.max_samples]
        else:
            wav = np.pad(wav, (0, self.max_samples - len(wav)))
        # Normalise
        wav = wav / (np.abs(wav).max() + 1e-8)
        return {'input_values': torch.tensor(wav, dtype=torch.float32),
                'labels': torch.tensor(cluster, dtype=torch.long)}

train_ds = RavdessDataset(train)
val_ds   = RavdessDataset(val)
test_ds  = RavdessDataset(test)

# Class-weighted sampler for train
train_labels = np.array([s[1] for s in train])
class_counts = np.bincount(train_labels, minlength=4)
weights = 1.0 / class_counts[train_labels]
sampler = WeightedRandomSampler(weights, num_samples=len(train_labels), replacement=True)

BATCH = 4   # wav2vec2-large needs small batch on T4 16 GB
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('Loaders ready. Train batches/epoch:', len(train_loader))

## 5 · Model — MindSyncAudioModel (wav2vec2-large + classifier)

In [ ]:
import torch, torch.nn as nn
from transformers import Wav2Vec2Model

class Wav2VecAudioEncoder(nn.Module):
    def __init__(self, model_name='facebook/wav2vec2-large-960h', dropout=0.1, freeze_feature=True):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size  # 1024
        if freeze_feature:
            if hasattr(self.encoder, 'freeze_feature_encoder'):
                self.encoder.freeze_feature_encoder()
            else:
                for p in self.encoder.feature_extractor.parameters():
                    p.requires_grad = False
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_values):
        out = self.encoder(input_values=input_values).last_hidden_state  # (B, T, 1024)
        e = out.mean(dim=1)                                              # (B, 1024)
        return self.dropout(e)

class AudioClassificationHead(nn.Module):
    def __init__(self, hidden_size=1024, num_classes=4, dropout=0.1):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes),
        )
    def forward(self, e):
        return self.classifier(e)

class MindSyncAudioModel(nn.Module):
    def __init__(self, model_name='facebook/wav2vec2-large-960h', num_classes=4):
        super().__init__()
        self.encoder = Wav2VecAudioEncoder(model_name=model_name)
        self.classifier = AudioClassificationHead(self.encoder.hidden_size, num_classes)
    def forward(self, input_values):
        emb = self.encoder(input_values)
        return {'embedding': emb, 'logits': self.classifier(emb)}

device = torch.device('cuda')
model = MindSyncAudioModel().to(device)
print('Params (M):', sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6)

## 6 · Train (5 epochs, mixed precision, best-F1 checkpoint)

In [ ]:
import torch, time, numpy as np
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score

EPOCHS  = 5
LR_BACK = 1e-5  # wav2vec2 backbone
LR_HEAD = 1e-3  # classifier head

optim = AdamW([
    {'params': model.encoder.parameters(),    'lr': LR_BACK},
    {'params': model.classifier.parameters(), 'lr': LR_HEAD},
], weight_decay=1e-4)

scaler = torch.cuda.amp.GradScaler()
criterion = nn.CrossEntropyLoss()

def evaluate(loader):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for batch in loader:
            iv = batch['input_values'].to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits = model(iv)['logits']
            preds.extend(logits.argmax(-1).cpu().tolist())
            gts.extend(batch['labels'].tolist())
    return accuracy_score(gts, preds), f1_score(gts, preds, average='macro')

best_f1 = 0.0
BEST_CKPT = CKPT_DIR / 'best_audio_model.pt'
for ep in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time(); total_loss = 0.0
    for i, batch in enumerate(train_loader):
        iv = batch['input_values'].to(device, non_blocking=True)
        lb = batch['labels'].to(device, non_blocking=True)
        optim.zero_grad()
        with torch.cuda.amp.autocast():
            out = model(iv)
            loss = criterion(out['logits'], lb)
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim)
        scaler.update()
        total_loss += loss.item()
        if (i + 1) % 50 == 0:
            print(f'  ep{ep} step {i+1}/{len(train_loader)} loss {total_loss/(i+1):.4f}', flush=True)
    val_acc, val_f1 = evaluate(val_loader)
    marker = ''
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({'model_state_dict': model.state_dict(),
                    'epoch': ep, 'val_acc': val_acc, 'val_f1': val_f1,
                    'cluster_names': CLUSTER_NAMES},
                   BEST_CKPT)
        marker = ' ✓ Saved'
    print(f'Epoch {ep}: loss {total_loss/len(train_loader):.4f} | val acc {val_acc:.4f} | val F1 {val_f1:.4f} | {time.time()-t0:.0f}s{marker}', flush=True)

# Final test set evaluation
ckpt = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
test_acc, test_f1 = evaluate(test_loader)
print(f'\n🏆 FINAL TEST: acc {test_acc:.4f} | F1 {test_f1:.4f}')
print(f'   Best val ep: {ckpt["epoch"]} | val F1: {ckpt["val_f1"]:.4f}')

## 7 · Upload best checkpoint to HF Model repo

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
REPO = 'Ubaida1/mindsync-audio-model'
api.create_repo(repo_id=REPO, repo_type='model', exist_ok=True)
api.upload_file(
    path_or_fileobj=str(BEST_CKPT),
    path_in_repo='best_audio_model.pt',
    repo_id=REPO, repo_type='model',
    commit_message=f'Stage 2: wav2vec2-Large, test acc {test_acc:.4f}, F1 {test_f1:.4f}',
)
print('Uploaded → https://huggingface.co/' + REPO)

## Done!

Now tell Claude **"Stage 2 done"** with the printed `test acc` / `F1`. He'll:
1. Update the HF Space to also serve the audio model
2. Verify predictions on the cloud backend
3. Prepare **Stage 3: Multimodal Fusion (CMAF)**